# Machine-checked reproduction of the flood-kernel result

**What this notebook shows.** A finite-state machine (Seppa) running on a
Raspberry Pi 5 optimizes a flood-simulation GPU kernel under three physics
gates. The machine, not the proposing model, measures the baseline, verifies
each variant, benchmarks it, and issues the keep-or-revert verdict; a
benchmark for an unverified kernel is unreachable by construction. This
notebook derives, from the committed raw transcripts:

1. the machine's own baseline-and-keep ledger for the fused strip-2 kernel;
2. the server refusing a benchmark request for a kernel that failed the gate;
3. a five-run repeatability campaign scored against a fixed pass criterion;
4. the ledger of a session in which a language model drove the loop.

**Data provenance.** All inputs are unmodified transcripts under
`../docs/paper/artifacts/`, produced on the board itself:

| Directory / file | Produced by | Date |
|---|---|---|
| `mcp_flood2_recreation_2026-07-11.jsonl` | `drive_flood2_mcp.py` against `theodosia_server.py --http --flood2` | 2026-07-11 |
| `passk_2026-08-09/` | `passk_flood2.py` (5 scored runs, cool starts) | 2026-08-09 |
| `claude_sessions/20260802T150755Z/` | `claude_driver.sh` (model-driven session, budget 3) | 2026-08-02 |

This notebook supports Section VI of `../docs/paper/paper.pdf`. It runs
offline: nothing here needs the Pi.

In [1]:
import json
from pathlib import Path

import analysis_utils as au

ART = Path("../docs/paper/artifacts")
transcript = ART / "mcp_flood2_recreation_2026-07-11.jsonl"
events = [json.loads(ln) for ln in transcript.read_text().splitlines() if ln.startswith("{")]
[e.get("action") for e in events if "action" in e]

['characterize',
 'baseline',
 'hypothesize',
 'implement',
 'compile_',
 'verify',
 'benchmark',
 'evaluate',
 'log_variant',
 'hypothesize',
 'implement',
 'compile_',
 'verify',
 'benchmark',
 'log_variant']

## 1. The machine re-derives the result

The client submits the fused strip-2 kernel; every measurement and the
verdict below come from the server.

In [2]:
for e in events:
    r = e.get("result", {})
    inner = r.get("result", {}) if isinstance(r, dict) else {}
    if e.get("action") == "baseline":
        print("baseline:", inner)
    if e.get("action") == "log_variant" and isinstance(inner.get("logged"), dict):
        print("ledger:  ", json.dumps(inner["logged"]))

baseline: {'baseline_steps_per_sec': 1346.8013468013469, 'gate_ok': True}
ledger:   {"exp": 1, "fused": true, "strip": 2, "compile_ok": true, "verify_ok": true, "steps_per_sec": 2116.4021164021165, "best_sps": 2116.4021164021165, "verdict": "keep"}
ledger:   {"exp": 2, "fused": true, "strip": 2, "compile_ok": true, "verify_ok": false, "steps_per_sec": null, "best_sps": 2116.4021164021165, "verdict": "revert"}


Experiment 1 is kept at 2,116.4 steps/s against the machine's own
1,346.8 baseline (1.57x). Experiment 2 is the deliberately broken kernel
(rainfall doubled in one of two cell updates): it compiles, fails the
mass-conservation gate, and is ledgered as a revert with a null
`steps_per_sec`.

## 2. The gate refusal

After the failed gate, the client requests `benchmark` anyway. The
server's verbatim answer:

In [3]:
for e in events:
    if (
        e.get("action") == "benchmark"
        and isinstance(e.get("result"), dict)
        and e["result"].get("error") == "invalid_transition"
    ):
        print(json.dumps(e["result"], indent=1))

{
 "error": "invalid_transition",
 "requested": "benchmark",
 "valid_next_actions": [
  "log_variant"
 ],
 "next_action_schemas": {
  "log_variant": {}
 },
 "message": "action 'benchmark' is not reachable from current state. Valid actions now: ['log_variant'].",
 "next_hint": "Action 'benchmark' is not reachable from the current state. Reachable now: log_variant."
}


## 3. Repeatability: pass^k

`passk_flood2.py` repeats the whole two-cycle reproduction from cool
starts and scores each run against a fixed criterion: baseline gates
green inside 1,300 to 1,400 steps/s, the fused kernel kept at 1.5x or
better, and the broken kernel refused and nulled.

In [4]:
summary = json.loads((ART / "passk_2026-08-09" / "summary.json").read_text())
for r in summary["runs"]:
    print(
        f"run {r['run']}: pass={r['pass']}  baseline={r['baseline']:.1f}  "
        f"kept={r['kept']:.1f}  speedup={r['kept'] / r['baseline']:.3f}"
    )
print()
print(summary["summary"])
assert summary["summary"]["pass_k"], "pass^k failed"
print(
    "\npass^{} = {}/{}".format(
        summary["summary"]["k"], summary["summary"]["passes"], summary["summary"]["k"]
    )
)

run 1: pass=True  baseline=1351.4  kept=2127.7  speedup=1.574
run 2: pass=True  baseline=1342.3  kept=2127.7  speedup=1.585
run 3: pass=True  baseline=1351.4  kept=2127.7  speedup=1.574
run 4: pass=True  baseline=1351.4  kept=2127.7  speedup=1.574
run 5: pass=True  baseline=1346.8  kept=2127.7  speedup=1.580

{'k': 5, 'passes': 5, 'pass_k': True, 'baseline_mean': 1348.6, 'baseline_stdev': 4.1, 'kept_mean': 2127.7, 'kept_stdev': 0.0, 'speedups': [1.574, 1.585, 1.574, 1.574, 1.58]}

pass^5 = 5/5


## 4. A language model drives the loop

One session was driven end to end by a language model over MCP with a
three-experiment budget (proposer, client, and settings are recorded in
the session's `run_meta.txt`). The server's `characterize` payload names
the stencil's bound and exposes the strip parameter, so this session
exercises the loop rather than blind discovery. Its ledger:

In [5]:
import re

sess = ART / "claude_sessions" / "20260802T150755Z"
print((sess / "run_meta.txt").read_text())
raw = (sess / "ledger_responses.json").read_text()
seen = set()
for m in re.finditer(r'\\?"logged\\?":\s*\{(.*?)\}', raw.replace('\\"', '"')):
    entry = "{" + m.group(1) + "}"
    if entry not in seen:
        seen.add(entry)
        print(entry)

date_utc=20260802T150755Z
mcp_url=http://pi.local:8000/mcp
model=claude-sonnet-5
effort=high
max_experiments=3
temperature=not exposed by the Claude Code client (server default)
auth=signed-in Claude subscription session
client=2.1.220 (Claude Code)

{"exp":1,"fused":false,"strip":4,"compile_ok":true,"verify_ok":true,"steps_per_sec":1470.5882352941176,"best_sps":1470.5882352941176,"verdict":"keep"}
{"exp":2,"fused":false,"strip":8,"compile_ok":true,"verify_ok":true,"steps_per_sec":921.6589861751152,"best_sps":1470.5882352941176,"verdict":"revert"}
{"exp":3,"fused":false,"strip":6,"compile_ok":true,"verify_ok":true,"steps_per_sec":1298.7012987012988,"best_sps":1470.5882352941176,"verdict":"revert"}


Strip 4 is kept (+8.8% over the session's own baseline); strips 8 and 6
are measured and reverted by the machine; fusion is never proposed within
the budget.

## Rerunning live

```
# on the Pi
.venv/bin/python theodosia_server.py --http --flood2
# from any machine on the network
.venv/bin/python drive_flood2_mcp.py http://<pi>:8000/mcp
python3 passk_flood2.py 5 http://<pi>:8000/mcp
./claude_driver.sh http://<pi>:8000/mcp
```

## Environment

In [6]:
au.environment()

python 3.14.3
matplotlib 3.11.1


nbformat 5.11.0
